# Colab Training - Hybrid BiLSTM-CRF POS Tagger

Notebook ini menjalankan eksperimen BiLSTM-CRF di Colab GPU.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'POSTagPantun'
REPO_URL = 'https://github.com/NazirArifin/pantunify.git'
REPO_DIR = PROJECT_PARENT / 'pantunify'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists, skip clone.')

%cd {REPO_DIR}
!git --no-pager status -sb


/content/drive/MyDrive/POSTagPantun/pantunify
## main...origin/main


In [ ]:
import subprocess
import sys

PYTHON = sys.executable
print('Python executable:', PYTHON)
subprocess.run([PYTHON, '-V'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch', 'torchcrf'], check=True)


Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.7 MB/s eta 0:00:00


In [4]:
import json

DATA_DIR = REPO_DIR / 'data' / 'article5'
TRAIN_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-train.conllu'
DEV_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-dev.conllu'
TRAIN_CSUI = DATA_DIR / 'id_csui-ud-train.conllu'
OUT_DIR = DATA_DIR / 'processed_crf'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_GSD, DEV_GSD, TRAIN_CSUI]:
    print(p, '->', 'OK' if p.exists() else 'MISSING')


/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/UD_Indonesian-GSD/id_gsd-ud-train.conllu -> OK
/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/UD_Indonesian-GSD/id_gsd-ud-dev.conllu -> OK
/content/drive/MyDrive/POSTagPantun/pantunify/data/article5/id_csui-ud-train.conllu -> OK


In [5]:
CONVERTER = REPO_DIR / 'scripts' / 'article5' / 'convert_conllu_to_jsonl.py'
subprocess.run(
    [
        PYTHON,
        str(CONVERTER),
        '--train-conllu', str(TRAIN_GSD), str(TRAIN_CSUI),
        '--dev-conllu', str(DEV_GSD),
        '--out-dir', str(OUT_DIR),
    ],
    check=True,
)


[OK] train jsonl: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/train.jsonl
[OK] dev jsonl: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/dev.jsonl
[OK] vocab: /content/drive/MyDrive/POSTagPantun/pantunify/data/article5/processed_crf/vocab.json
[INFO] train_sentences=5138, dev_sentences=559
[INFO] word_vocab=20416, char_vocab=281, tags=17


In [6]:
VOCAB_PATH = OUT_DIR / 'vocab.json'
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    vocab = json.load(f)

num_chars = len(vocab['char_vocab'])
vocab_size = len(vocab['word_vocab'])
num_classes = len(vocab['tag_vocab'])
print(num_chars, vocab_size, num_classes)


281 20416 17


In [7]:
TRAIN_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'train_hybrid_bilstm_crf_pos.py'
TRAIN_JSONL = OUT_DIR / 'train.jsonl'
DEV_JSONL = OUT_DIR / 'dev.jsonl'
CKPT_DIR = REPO_DIR / 'scripts' / 'article5' / 'checkpoints_crf_colab'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        PYTHON,
        str(TRAIN_SCRIPT),
        '--train-jsonl', str(TRAIN_JSONL),
        '--valid-jsonl', str(DEV_JSONL),
        '--num-chars', str(num_chars),
        '--vocab-size', str(vocab_size),
        '--num-classes', str(num_classes),
        '--save-dir', str(CKPT_DIR),
        '--epochs', '20',
        '--batch-size', '64',
        '--word-emb-dim', '300',
        '--lr', '0.001',
        '--lr-crf', '0.003',
        '--dropout', '0.3',
        '--weight-decay', '1e-5',
        '--grad-clip', '1.0',
        '--best-metric', 'composite',
        '--score-weight-acc', '0.5',
        '--score-weight-oov-acc', '0.3',
        '--score-weight-critical-f1', '0.2',
        '--critical-tags', 'ADV,PART,NOUN,VERB',
        '--tag-vocab-json', str(VOCAB_PATH),
        '--vocab-json', str(VOCAB_PATH),
        '--enable-disemvowel-augmentation',
        '--disemvowel-tags', 'ADV,PART,PRON,CCONJ,SCONJ',
        '--disemvowel-prob', '0.5',
        '--early-stopping-patience', '4',
        '--metrics-jsonl', str(CKPT_DIR / 'metrics.jsonl'),
    ],
    check=True,
)


Traceback (most recent call last):
  File "/content/drive/MyDrive/POSTagPantun/pantunify/scripts/article5/train_hybrid_bilstm_crf_pos.py", line 10, in <module>
    from HybridBiLSTMCRFPOSTagger import HybridBiLSTMCRFPOSTagger
  File "/content/drive/MyDrive/POSTagPantun/pantunify/scripts/article5/HybridBiLSTMCRFPOSTagger.py", line 3, in <module>
    from torchcrf import CRF
ModuleNotFoundError: No module named 'torchcrf'


In [ ]:
import csv
import json
import subprocess
from pathlib import Path

import torch

SWEEP_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'run_crf_tuning_sweep.py'
EVAL_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'evaluate_checkpoint_on_conllu_crf.py'
PREDICT_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'predict_pantun_csv_crf.py'
SWEEP_OUT_DIR = REPO_DIR / 'scripts' / 'article5' / 'tuning_runs_crf_colab'
SWEEP_OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SWEEP = True  # ubah False jika ingin skip dan pakai checkpoint existing
ENABLE_DISEMVOWEL_AUG = True
DISEMVOWEL_TAGS = 'ADV,PART,PRON,CCONJ,SCONJ'
DISEMVOWEL_PROB = '0.5'

if RUN_SWEEP:
    sweep_cmd = [
        PYTHON,
        str(SWEEP_SCRIPT),
        '--train-jsonl', str(OUT_DIR / 'train.jsonl'),
        '--valid-jsonl', str(OUT_DIR / 'dev.jsonl'),
        '--vocab-json', str(VOCAB_PATH),
        '--out-root', str(SWEEP_OUT_DIR),
        '--epochs', '20',
        '--batch-size', '64',
        '--execute',
    ]
    if ENABLE_DISEMVOWEL_AUG:
        sweep_cmd.extend([
            '--enable-disemvowel-augmentation',
            '--disemvowel-tags', DISEMVOWEL_TAGS,
            '--disemvowel-prob', DISEMVOWEL_PROB,
        ])
    subprocess.run(sweep_cmd, check=True)

run_summaries = []
for run_dir in sorted(SWEEP_OUT_DIR.glob('run_*')):
    best_ckpt = run_dir / 'best.pt'
    if not best_ckpt.exists():
        continue
    obj = torch.load(best_ckpt, map_location='cpu')
    metrics = obj.get('metrics', {})
    run_summaries.append({
        'run': run_dir.name,
        'epoch': obj.get('epoch'),
        'selection_score': float(metrics.get('selection_score', metrics.get('accuracy', 0.0))),
        'accuracy': float(metrics.get('accuracy', 0.0)),
        'oov_accuracy': float(metrics.get('oov_accuracy', 0.0)),
        'critical_macro_f1': float(metrics.get('critical_macro_f1', 0.0)),
        'best_ckpt': str(best_ckpt),
    })

if not run_summaries:
    fallback = CKPT_DIR / 'best.pt'
    if not fallback.exists():
        raise RuntimeError('Tidak ada checkpoint dari sweep maupun single run')
    best_run = {
        'run': 'single_run_fallback',
        'best_ckpt': str(fallback),
    }
else:
    run_summaries = sorted(run_summaries, key=lambda x: x['selection_score'], reverse=True)
    print('Top tuning runs:')
    for r in run_summaries[:6]:
        print(r)
    best_run = run_summaries[0]
    leaderboard_path = SWEEP_OUT_DIR / 'leaderboard.json'
    with open(leaderboard_path, 'w', encoding='utf-8') as f:
        json.dump(run_summaries, f, ensure_ascii=False, indent=2)
    print('Saved leaderboard:', leaderboard_path)

BEST_CKPT = Path(best_run['best_ckpt'])
print('\nSelected best run:', best_run['run'])
print('Best checkpoint:', BEST_CKPT)

TEST_CONLLU = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-test.conllu'
result = subprocess.run(
    [
        PYTHON,
        str(EVAL_SCRIPT),
        '--checkpoint', str(BEST_CKPT),
        '--vocab-json', str(VOCAB_PATH),
        '--conllu', str(TEST_CONLLU),
        '--batch-size', '64',
    ],
    capture_output=True,
    text=True,
    check=True,
)
print('\n[Test set metrics]')
print(result.stdout)

PRED_OUT_100 = OUT_DIR / 'pantun_gaul_pred_100_crf.csv'
subprocess.run(
    [
        PYTHON,
        str(PREDICT_SCRIPT),
        '--input-csv', str(DATA_DIR / 'pantun_gaul.csv'),
        '--checkpoint', str(BEST_CKPT),
        '--vocab-json', str(VOCAB_PATH),
        '--output-csv', str(PRED_OUT_100),
        '--limit', '100',
        '--text-column', 'text_pantun',
    ],
    check=True,
)

print('\nPreview output 10 baris pertama:')
with open(PRED_OUT_100, 'r', encoding='utf-8', newline='') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(row)
        if i >= 10:
            break
print('\nOutput file:', PRED_OUT_100)
